In [ ]:
import os, sys
import pathlib
import json

import torch
import pandas as pd
from transformers import AutoTokenizer
import torch.nn.functional as F

from data.dataset import build_id_to_text
from data.preprocessing import run_preprocessing
from model.train import (
    train_stage1, train_stage2,
    evaluate_epoch, DEFAULT_CFG, format_metrics,
    run_hpo, run_hpo_reranker, build_item_embedding_cache, tokenise_batch,
    build_cf_mappings,
)

from model.architecture import ContrastiveEncoder, TwoTowerModel
from model.reranker import CrossEncoderReranker, TwoTowerWithReranker, train_stage3
from model.metrics import evaluate_reranker, format_metrics

In [2]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = pathlib.Path('checkpoints/')
output_dir.mkdir(exist_ok=True)

ENCODER = 'intfloat/multilingual-e5-base'
use_tower_hpo = True

# Data

In [4]:
os.makedirs('raw_data', exist_ok=True)

BASE_URL = 'https://huggingface.co/datasets/kdduha/shikimori-recsys/resolve/main'

for fname in ['anime.csv', 'genres.csv', 'users_rates.csv']:
    dest = f'raw_data/{fname}'
    if not os.path.exists(dest):
        !wget -q "{BASE_URL}/{fname}" -O "{dest}"
        print(f'Downloaded {fname}')
    else:
        print(f'Already have {fname}')

!ls -lh raw_data/

Already have anime.csv
Already have genres.csv
Already have users_rates.csv
total 15M
-rw-rw-r-- 1 s.senichev s.senichev 8.7M Mar  2 23:30 anime.csv
-rw-rw-r-- 1 s.senichev s.senichev 2.9K Mar  2 23:30 genres.csv
-rw-rw-r-- 1 s.senichev s.senichev 6.0M Mar  2 23:30 users_rates.csv


In [5]:
stats = run_preprocessing(
    data_dir='raw_data',
    output_dir='processed_data',
)
print(json.dumps(stats, indent=2))

15:53:19  INFO      Loading raw CSVs from raw_data
15:53:19  INFO      Raw sizes — anime: 9950  genres: 80  interactions: 67071
15:53:21  INFO      Anime processing done. Non-null text_input: 9950 / 9950
15:53:21  INFO      Saved anime_processed.parquet
15:53:21  INFO      Saved genre_vocab.json  (80 genres)
15:53:22  INFO      Dropped 0 rows with unparseable anime_id
15:53:22  INFO      Interactions — explicit (scored): 28129  implicit (watched, unscored): 38677  dropped (score=0, episodes=0): 265
15:53:23  INFO      Dropped 1515 interactions for unknown anime_id
15:53:23  INFO      Removed users with < 3 explicit ratings. Kept: 531 users  39345 rows
15:53:23  INFO      Final — 39345 rows (27214 explicit, 12131 implicit)  531 users  3695 items
15:53:23  INFO      Temporal split — train: 38393  val: 476  test: 476  (eligible users: 476 / 531)
15:53:23  WARNING   Temporal leakage: 130 val rows have created_at BEFORE user's latest train item (likely duplicate timestamps)
15:53:23  WARNIN

{
  "n_anime_raw": 9950,
  "n_anime_processed": 9950,
  "n_genres": 80,
  "n_interactions_raw": 67071,
  "n_interactions_clean": 39345,
  "n_users": 531,
  "n_items": 3695,
  "train_size": 38393,
  "val_size": 476,
  "test_size": 476,
  "score_mean": 5.498,
  "score_std": 3.939,
  "density_pct": 2.0053
}


In [7]:
anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
train_df = pd.read_parquet('processed_data/train_interactions.parquet')
val_df = pd.read_parquet('processed_data/val_interactions.parquet')
test_df = pd.read_parquet('processed_data/test_interactions.parquet')

print('Anime sample')
display(anime_df[['id','name','genre_names','rating_ordinal','score_global','text_input']].head(3))

print('\nInteraction sample')
display(train_df.head(3))

print(f'\nTrain: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
# print(f'Score distribution:\n{train_df.score_raw.value_counts().sort_index()}')

Anime sample


,id,name,genre_names,rating_ordinal,score_global,text_input
0,52991,Sousou no Frieren,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.921111,Sousou no Frieren (Провожающая в последний пут...
1,59978,Sousou no Frieren 2nd Season,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.910000,Sousou no Frieren 2nd Season (Провожающая в по...
2,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy, Shounen, M...",3.0,0.900000,Fullmetal Alchemist: Brotherhood (Стальной алх...



Interaction sample


,user_id,anime_id,score_raw,score_norm,rewatches,episodes,completion_rate,confidence,is_explicit,created_at
0,1721273,5114,9,0.888889,0,64,1.0,0.888889,True,2025-11-07 19:14:34+00:00
1,1721273,9253,10,1.000000,0,24,1.0,1.000000,True,2025-11-07 19:15:22+00:00
2,1721273,2001,10,1.000000,0,27,1.0,1.000000,True,2025-11-07 19:18:22+00:00



Train: 38,393  Val: 476  Test: 476


# Model

### Load checkpoint if it exists

In [ ]:
# output_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')

# with open(output_dir / 'config.json') as f:
#     cfg = json.load(f)

# tokenizer = AutoTokenizer.from_pretrained(str(output_dir))

# # Rebuild CF mappings
# item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
# cf_dim = cfg.get('cf_dim', 0)
# if cf_dim == 0:
#     n_items_cf = 0
#     n_users_cf = 0

# model = TwoTowerModel(
#     encoder_name=ENCODER,
#     proj_dim=cfg['proj_dim'],
#     nhead=cfg['nhead'],
#     temperature=cfg['temperature'],
#     dropout=cfg['dropout'],
#     freeze_mode=cfg['freeze_mode'],
#     lora_rank=cfg['lora_rank'],
#     lora_alpha=cfg.get('lora_alpha', 16.0),
#     n_items=n_items_cf,
#     n_users=n_users_cf,
#     cf_dim=cf_dim,
# )

# state = torch.load(output_dir / 'model.pt', map_location=device)
# model.load_state_dict(state)
# model.to(device)
# model.eval()
# print('Final model loaded')

## Train main model

### HPO

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
hpo_path = output_dir / 'hpo_best_params.json'

if use_tower_hpo:
    print('Starting HPO')
    best_params = run_hpo(
        train_df=train_df,
        val_df=val_df,
        anime_df=anime_df,
        base_cfg=DEFAULT_CFG,
        output_dir=pathlib.Path('checkpoints'),
        device=device,
        n_trials=20,
        encoder_name=ENCODER,
    )
    
    with open(hpo_path) as f:
        hpo_data = json.load(f)
    cfg = {**DEFAULT_CFG, **hpo_data['best_params']}
    print(f'Using HPO params (NDCG@10={hpo_data["best_value"]:.4f})')

else:
    cfg = {**DEFAULT_CFG}
    print('Using default hyperparameters')

print(json.dumps(cfg, indent=2))

### Train Stage 1 (item tower)

In [ ]:
s1_model = ContrastiveEncoder(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
)

s1_model = train_stage1(
    s1_model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

torch.save(s1_model.tower.state_dict(), output_dir / 'stage1_encoder.pt')
print('Stage 1 complete')

### Train Stage 2 (user tower)

In [ ]:
# Build CF mappings
item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
cf_dim = cfg.get('cf_dim', 0)
if cf_dim == 0:
    n_items_cf = 0
    n_users_cf = 0

model = TwoTowerModel(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    nhead=cfg['nhead'],
    temperature=cfg['temperature'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
    n_items=n_items_cf,
    n_users=n_users_cf,
    cf_dim=cf_dim,
)

state = torch.load(output_dir / 'stage1_encoder.pt', map_location='cpu')
model.item_tower.load_state_dict(state, strict=False)
print('Stage 1 weights loaded into item tower')

model, best_val = train_stage2(
    model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

In [32]:
print('Best Val Metrics:', format_metrics(best_val))

Best Val Metrics: HR@10: 0.0315  HR@20: 0.0357  HR@5: 0.0168  MRR: 0.0150  NDCG@10: 0.0162  NDCG@20: 0.0173  NDCG@5: 0.0116


### Save model

In [11]:
final_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')
final_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), final_dir / 'model.pt')
tokenizer.save_pretrained(str(final_dir))
with open(final_dir / 'config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Model saved to {final_dir}')

Model saved to checkpoints/final_model_v4_with_rerank_hpo


## Evaluate model

In [12]:
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_epoch(
    model, tokenizer, test_df, anime_df,
    cfg=cfg, device=device, ks=[10, 20, 50],
    train_df=train_and_val,
)

print('TEST RESULTS')
for k, v in sorted(test_metrics.items()):
    print(f'{k:<12} {v:.4f}')

/home/s.senichev/anirec/model/train.py:240: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
/home/s.senichev/anirec/model/train.py:511: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
18:06:36  INFO      Evaluation: 476 / 476 users have context embeddings


TEST RESULTS
HR@10        0.0210
HR@20        0.0315
HR@50        0.0630
MRR          0.0085
NDCG@10      0.0089
NDCG@20      0.0115
NDCG@50      0.0177


## Reranker model

### Load checkpoint if it exists

In [36]:
# cfg = {**DEFAULT_CFG}

# s3_tokenizer = AutoTokenizer.from_pretrained(cfg["s3_encoder"])

# reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
# reranker_model.load_state_dict(
#     torch.load('checkpoints/stage3_best.pt', map_location='cpu')
# )
# reranker_model.to(device).eval()
# print("Reranker loaded")

### Train reranker with HPO

In [ ]:
from model.train import run_hpo_reranker, DEFAULT_CFG

best_s3_params = run_hpo_reranker(
    two_tower_model=model,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    base_cfg=cfg,
    output_dir=output_dir,
    device=device,
    n_trials=20,
)
print(best_s3_params)

In [ ]:
cfg_s3 = {**cfg, **best_s3_params}

s3_tokenizer = AutoTokenizer.from_pretrained(cfg_s3["s3_encoder"])
reranker_model = train_stage3(
    CrossEncoderReranker(encoder_name=cfg['s3_encoder']),
    s3_tokenizer, train_df, val_df, anime_df,
    cfg=cfg_s3, output_dir=output_dir, device=device,
)

### Save reranker model

In [21]:
torch.save(reranker_model.state_dict(), final_dir / 'reranker.pt')
s3_tokenizer.save_pretrained(str(final_dir / 'reranker_tokenizer'))

print(f'Reranker saved to {final_dir}')

Reranker saved to checkpoints/final_model_v4_with_rerank_hpo


In [ ]:
recommender = TwoTowerWithReranker(
    two_tower=model,
    reranker=reranker_model,
    tokenizer=tokenizer,
    reranker_tokenizer=s3_tokenizer,
    anime_df=anime_df,
    device=device,
    item_id_to_cf_idx=item_id_to_cf_idx if model.has_cf else None,
    user_id_to_cf_idx=user_id_to_cf_idx if model.has_cf else None,
)

### Evaludate reranker

In [24]:
val_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=val_df,
    train_df=train_df,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER VAL METRICS")
print(format_metrics(val_metrics))

22:18:53  INFO      Evaluating reranker on 476 users (retrieval_k=100)
22:18:54  INFO        32 / 476 users evaluated...
22:18:58  INFO        352 / 476 users evaluated...
22:19:00  INFO      Reranker eval done. Target in candidates: 67/476 (14.1%)


RERANKER VAL METRICS
HR@10: 0.0525  HR@20: 0.0924  HR@50: 0.1197  MRR: 0.1967  NDCG@10: 0.0305  NDCG@20: 0.0408  NDCG@50: 0.0462  retrieval_recall: 0.1408


In [25]:
# Test set (context = train + val)
import pandas as pd
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=test_df,
    train_df=train_and_val,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER TEST METRICS")
print(format_metrics(test_metrics))

22:19:00  INFO      Evaluating reranker on 476 users (retrieval_k=100)
22:19:00  INFO        32 / 476 users evaluated...
22:19:05  INFO        352 / 476 users evaluated...
22:19:07  INFO      Reranker eval done. Target in candidates: 63/476 (13.2%)


RERANKER TEST METRICS
HR@10: 0.0378  HR@20: 0.0714  HR@50: 0.1239  MRR: 0.1517  NDCG@10: 0.0211  NDCG@20: 0.0294  NDCG@50: 0.0400  retrieval_recall: 0.1324


# Load fully trained model

In [ ]:
final_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(final_dir / 'config.json') as f:
    cfg = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(str(final_dir))

# Rebuild CF mappings for the loaded model
anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
train_df = pd.read_parquet('processed_data/train_interactions.parquet')
item_id_to_cf_idx, user_id_to_cf_idx, n_items_cf, n_users_cf = build_cf_mappings(anime_df, train_df)
cf_dim = cfg.get('cf_dim', 0)
if cf_dim == 0:
    n_items_cf = 0
    n_users_cf = 0

model = TwoTowerModel(
    encoder_name='intfloat/multilingual-e5-base',
    proj_dim=cfg['proj_dim'], nhead=cfg['nhead'],
    temperature=cfg['temperature'], dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'], lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
    n_items=n_items_cf,
    n_users=n_users_cf,
    cf_dim=cf_dim,
)
model.load_state_dict(torch.load(final_dir / 'model.pt', map_location='cpu'))
model.to(device).eval()

s3_tokenizer = AutoTokenizer.from_pretrained(str(final_dir / 'reranker_tokenizer'))
reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
reranker_model.load_state_dict(torch.load(final_dir / 'reranker.pt', map_location='cpu'))
reranker_model.to(device).eval()

print('Loaded')

# Inference 

In [29]:
# sample user

user_history = [
    (5114, 10),   # FMA Brotherhood
    (9253, 9),   # Steins;Gate
    (11061, 9),   # HxH 2011
    (37510, 8),   # Mob Psycho 100 II
    (31240, 3),   # Some disliked item (placeholder)
]
top_k = 20

## Test model without reranker

In [ ]:
model.eval()

item_matrix, id_to_idx = build_item_embedding_cache(
    model, tokenizer, anime_df, cfg, device,
    item_id_to_cf_idx=item_id_to_cf_idx if model.has_cf else None,
)
idx_to_id = {v: k for k, v in id_to_idx.items()}

context_ids = [aid  for aid, _ in user_history if aid in id_to_idx]
context_scores = [s/10 for aid, s  in user_history if aid in id_to_idx]
watched_ids = set(context_ids)

ctx_idxs = torch.tensor([id_to_idx[a] for a in context_ids], dtype=torch.long)
ctx_embs = item_matrix[ctx_idxs].unsqueeze(0).to(device)
ctx_scores= torch.tensor([context_scores], dtype=torch.float).to(device)
ctx_mask  = torch.ones(1, len(context_ids), dtype=torch.bool).to(device)

with torch.no_grad():
    # No user_idx for anonymous/cold-start user — falls back to content-only
    user_vec = model.encode_user(ctx_embs, ctx_scores, ctx_mask)
    scores = (user_vec @ item_matrix.to(device).T).squeeze(0)

# Exclude watched items
watched_idxs = set(id_to_idx[a] for a in watched_ids if a in id_to_idx)
for idx in watched_idxs:
    scores[idx] = -1e9

top_idxs = scores.topk(top_k).indices.cpu().tolist()
top_ids = [idx_to_id[i] for i in top_idxs]

print(f'Top-{top_k} recommendations for this user:\n')
id_to_row = anime_df.set_index('id')
for rank, aid in enumerate(top_ids, 1):
    row = id_to_row.loc[aid]
    genres = ', '.join(row['genre_names'][:3])
    print(f'  {rank:2}. {row["name"]:<45} [{genres}]  score={row["score_global"]:.2f}')

## Test model with reranker

In [31]:
recs = recommender.recommend(user_history, top_k=top_k, retrieval_k=100)

print(f'Top- recommendations\n')
print(f'  {"#":<4} {"Title":<45} {"Reranker":>10} {"Retrieved":>10} {"Genres"}')
print('  ' + '-'*85)
for r in recs:
    genres = ', '.join(list(r['genres'])[:3]) if len(r['genres']) > 0 else '—'
    moved = r['two_tower_rank'] - r['rank']
    arrow = f'↑{moved}' if moved > 0 else (f'↓{-moved}' if moved < 0 else '=')
    print(f"  {r['rank']:<4} {r['name']:<45} {r['reranker_score']:>6.3f}  "
          f"#{r['two_tower_rank']:<3} {arrow:<5}  [{genres}]")

Top- recommendations

  #    Title                                           Reranker  Retrieved Genres
  -------------------------------------------------------------------------------------
  1    Mob Psycho 100 III                             0.864  #54  ↑53    [Action, Comedy, Supernatural]
  2    Sen to Chihiro no Kamikakushi                  0.844  #78  ↑76    [Adventure, Mythology, Fantasy]
  3    One Piece                                      0.842  #74  ↑71    [Action, Adventure, Fantasy]
  4    Code Geass: Hangyaku no Lelouch R2             0.832  #8   ↑4     [Drama, Mecha, Sci-Fi]
  5    Kusuriya no Hitorigoto 2nd Season              0.831  #4   ↓1     [Mystery, Drama, Historical]
  6    Fullmetal Alchemist: The Conqueror of Shamballa  0.829  #83  ↑77    [Action, Adventure, Drama]
  7    Takopii no Genzai                              0.828  #13  ↑6     [Drama, Sci-Fi, Shounen]
  8    Bocchi the Rock!                               0.822  #16  ↑8     [Comedy, Music, CGDCT]
  9